# Fourier Transforms & Martingale Pricing: A Computational Finance Notebook

---

## Abstract

This notebook provides an end-to-end demonstration of Fourier-based option pricing methods applied to a
zoo of stochastic models spanning the affine jump-diffusion and Lévy process families. We verify the
risk-neutral martingale constraint, visualise characteristic functions and risk-neutral densities, benchmark
Carr-Madan FFT and COS pricing against Black-Scholes closed forms, calibrate Heston and Variance Gamma
models to synthetic market data, and compute Fourier-based Greeks and Value-at-Risk.

---

## Table of Contents

| § | Section | Key Output |
|---|---------|------------|
| 0 | Preamble & Environment | matplotlib theme, palette |
| 1 | Model Instantiation | 6 models, martingale table |
| 2 | Martingale Verification | Fig 1 — bar chart of errors |
| 3 | Characteristic Functions | Fig 2 — CF grid; Fig 3 — modulus decay |
| 4 | Implied Volatility Smiles | Fig 4 — smile overlay; Fig 5 — decomposition |
| 5 | Risk-Neutral Density | Fig 6 — density overlay; Fig 7 — tail comparison |
| 6 | Carr-Madan FFT Pricing | Fig 8 — FFT validation |
| 7 | COS Method Convergence | Fig 9 — convergence study |
| 8 | Volatility Surface Calibration | Fig 10 — fit quality; Fig 11 — residuals |
| 9 | Fourier Risk Analytics | Fig 12 — VaR & Greeks dashboard |
| 10 | Cumulant Analysis | Fig 13 — cumulants; Fig 14 — summary heatmap |

---

> **Models covered:** Black-Scholes-Merton · Heston Stochastic Volatility · Variance Gamma ·
> CGMY (Tempered Stable) · Normal Inverse Gaussian · Merton Jump-Diffusion
>
> **Pricing methods:** Carr-Madan FFT (1999) · Fang-Oosterlee COS (2008) · Gil-Pelaez inversion

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import PercentFormatter, FuncFormatter
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':    '#0D1B35',
    'axes.facecolor':      '#1B3A6B',
    'axes.edgecolor':      '#A8BFCE',
    'axes.labelcolor':     '#FFFFFF',
    'axes.titlecolor':     '#00A0DC',
    'xtick.color':         '#A8BFCE',
    'ytick.color':         '#A8BFCE',
    'text.color':          '#FFFFFF',
    'grid.color':          '#2C4A7A',
    'grid.linewidth':      0.6,
    'grid.alpha':          0.7,
    'legend.facecolor':    '#0D1B35',
    'legend.edgecolor':    '#0070C0',
    'legend.labelcolor':   '#A8BFCE',
    'font.family':         'monospace',
    'font.size':           11,
    'axes.titlesize':      14,
    'axes.labelsize':      12,
    'figure.titlesize':    16,
    'lines.linewidth':     2.0,
    'lines.antialiased':   True,
})

COLOR_PALETTE = {
    'navy':   '#0D1B35',
    'blue':   '#1B3A6B',
    'teal':   '#0070C0',
    'lteal':  '#00A0DC',
    'white':  '#FFFFFF',
    'gray':   '#A8BFCE',
    'gold':   '#F5A623',
    'green':  '#27AE60',
    'red':    '#C0392B',
    'purple': '#8E44AD',
    'orange': '#E67E22',
}

MODEL_COLORS = {
    'BSM':    '#A8BFCE',
    'Heston': '#0070C0',
    'VG':     '#F5A623',
    'CGMY':   '#27AE60',
    'NIG':    '#8E44AD',
    'MJD':    '#E74C3C',
}

print("Environment configured successfully.")
print(f"NumPy {np.__version__} | Pandas {pd.__version__} | Matplotlib {plt.matplotlib.__version__}")

---
## Section 1 — Environment Setup and Model Instantiation

This section imports the engine module and instantiates all six stochastic models using canonical
parameters representative of a liquid equity option market. The parameters have been chosen to produce
distinct but overlapping implied volatility profiles, enabling visual comparison across the model zoo.

**Market conventions used throughout:**
- Spot price **S = 100**, risk-free rate **r = 5%**, dividend yield **q = 2%**, maturity **T = 1 year**
- Forward price **F = S · exp((r−q)·T)**

Each model is verified against the risk-neutral martingale constraint immediately upon instantiation.
The constraint requires that the characteristic function evaluated at **u = −i** equals **exp((r−q)T)**,
which ensures that the discounted asset price is a true martingale under the pricing measure.

In [ ]:
import sys
sys.path.insert(0, '.')

from fourier_martingale import (
    BlackScholesMerton, HestonModel, VarianceGammaModel,
    CGMYModel, NormalInverseGaussianModel, MertonJumpDiffusion,
    MarketData, CalibrationEngine, FourierRiskAnalytics,
    MartingaleDiagnostics, bsm_call, implied_vol
)

S, r, q, T = 100.0, 0.05, 0.02, 1.0
F = S * np.exp((r - q) * T)

models = {
    'BSM':    BlackScholesMerton(S, r, q, T, sigma=0.20),
    'Heston': HestonModel(S, r, q, T, v0=0.04, kappa=2.0, theta=0.04, xi=0.3, rho=-0.70),
    'VG':     VarianceGammaModel(S, r, q, T, sigma=0.20, nu=0.5, theta=-0.10),
    'CGMY':   CGMYModel(S, r, q, T, C=1.0, G=5.0, M=10.0, Y=0.7),
    'NIG':    NormalInverseGaussianModel(S, r, q, T, alpha=15.0, beta=-5.0, delta=0.5),
    'MJD':    MertonJumpDiffusion(S, r, q, T, sigma=0.15, lambda_=0.5, mu_J=-0.05, delta_J=0.1),
}

print(f"{'Model':<20} {'Class':<35} {'Martingale OK'}")
print("-" * 65)
for name, m in models.items():
    ok = m.verify_martingale_condition()
    print(f"  {name:<18} {m.__class__.__name__:<35} {'PASS' if ok else 'FAIL'}")

---
## Section 2 — Martingale Property Verification

Under the risk-neutral measure **Q**, the discounted asset price process **e^{−rt} S_t** must be a
martingale. For models specified via their characteristic function **φ(u)** of the log-return
**X_T = log(S_T / S_0)**, this constraint translates to:

$$\varphi(-i) = \mathbb{E}^Q\left[e^{X_T}\right] = e^{(r-q)T}$$

**Drift correction (omega):** Most models include an explicit drift-correction term **ω** in the
log-characteristic function to enforce this equality. For example, in Black-Scholes:
$$\omega = -\frac{\sigma^2}{2}$$
and more generally **ω = −log(φ_0(−i))** where **φ_0** is the uncentred characteristic function.

**Gil-Pelaez inversion** (1951) provides a numerically stable route to recover probabilities from
the characteristic function via:
$$F(x) = \frac{1}{2} - \frac{1}{\pi} \int_0^\infty \text{Re}\left[\frac{e^{-iux} \varphi(u)}{iu}\right] du$$

The `MartingaleDiagnostics` class automates verification for all models simultaneously.

In [ ]:
diag_df = MartingaleDiagnostics.check_all_models(S, r, q, T)
print(diag_df.to_string())

In [ ]:
# --- Figure 1: Martingale Error by Model (horizontal bar, log scale) ---
tolerance = 1e-4
mart_errors = {}
for name, model in models.items():
    try:
        val = model.char_func(-1j)
        target = np.exp((r - q) * T)
        mart_errors[name] = abs(val - target)
    except Exception as e:
        print(f"  {name}: char_func(-1j) failed — {e}")
        mart_errors[name] = np.nan

model_names = list(mart_errors.keys())
error_vals  = [mart_errors[n] for n in model_names]
bar_colors  = [COLOR_PALETTE['green'] if (not np.isnan(e) and e < tolerance)
               else COLOR_PALETTE['red'] for e in error_vals]

fig, ax = plt.subplots(figsize=(12, 5))
y_pos = np.arange(len(model_names))
bars = ax.barh(y_pos, error_vals, color=bar_colors, edgecolor='#A8BFCE', linewidth=0.8, height=0.6)

ax.set_xscale('log')
ax.set_xlabel('| φ(−i) − exp((r−q)T) |', fontsize=12)
ax.set_yticks(y_pos)
ax.set_yticklabels(model_names, fontsize=11)
ax.set_title('Figure 1 — Martingale Error by Model (log scale)', fontsize=14)
ax.axvline(x=tolerance, color=COLOR_PALETTE['red'], linestyle='--',
           linewidth=1.5, label='Tolerance threshold (1e-4)')
ax.legend(loc='lower right')
ax.grid(True, axis='x', which='both')

for bar, err in zip(bars, error_vals):
    if not np.isnan(err) and err > 0:
        ax.text(err * 1.5, bar.get_y() + bar.get_height() / 2,
                f'{err:.2e}', va='center', ha='left',
                color=COLOR_PALETTE['white'], fontsize=9)

green_patch = mpatches.Patch(color=COLOR_PALETTE['green'], label='Error < 1e-4 (PASS)')
red_patch   = mpatches.Patch(color=COLOR_PALETTE['red'],   label='Error ≥ 1e-4 (FAIL)')
ax.legend(handles=[green_patch, red_patch,
          mpatches.Patch(color=COLOR_PALETTE['red'], label='Tolerance threshold (1e-4)',
                         fill=False, linestyle='--')],
          loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

---
## Section 3 — Characteristic Function Visualisation

The **characteristic function (CF)** φ(u) = E[e^{iuX}] is the central object of the Fourier pricing
framework. Because φ is the Fourier transform of the log-return density, Fourier inversion methods
such as Carr-Madan and COS can price European options directly from φ without ever simulating paths.

Key properties of φ under the risk-neutral measure:
- **φ(0) = 1** (normalisation)
- **φ(−i) = exp((r−q)T)** (martingale constraint)
- **|φ(u)| ≤ 1** for real u, with decay rate determined by the smoothness of the density
- The faster |φ(u)| decays, the smoother the density and the fewer Fourier terms needed for pricing

The real part Re[φ(u)] governs the symmetric component of the distribution, while Im[φ(u)] encodes
skewness. The modulus |φ(u)| = sqrt(Re²+Im²) reveals the effective bandwidth of each model.

In [ ]:
# --- Figure 2: 2x3 grid of characteristic function plots ---
u_range = np.linspace(0.01, 30, 500)

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

for idx, (name, model) in enumerate(models.items()):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])
    color = MODEL_COLORS[name]
    try:
        cf_vals = np.array([model.char_func(u) for u in u_range])
        ax.plot(u_range, cf_vals.real, color=color, linestyle='-',  linewidth=1.8, label='Re[φ(u)]')
        ax.plot(u_range, cf_vals.imag, color=color, linestyle='--', linewidth=1.4, alpha=0.75, label='Im[φ(u)]')
    except Exception as e:
        ax.text(0.5, 0.5, f'Error:\n{e}', transform=ax.transAxes,
                ha='center', va='center', color=COLOR_PALETTE['red'], fontsize=9)
    ax.axhline(0, color=COLOR_PALETTE['gray'], linewidth=0.6, linestyle=':')
    ax.set_title(name, fontsize=13, color=color)
    ax.set_xlabel('u', fontsize=10)
    ax.set_ylabel('φ(u)', fontsize=10)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True)

fig.suptitle(
    'Figure 2 — Characteristic Functions of Log-Returns Under Risk-Neutral Measure',
    fontsize=15, y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 3: CF modulus decay ---
u_range2 = np.linspace(0.01, 50, 1000)

fig, ax = plt.subplots(figsize=(12, 6))
for name, model in models.items():
    try:
        mods = np.array([abs(model.char_func(u)) for u in u_range2])
        ax.plot(u_range2, mods, color=MODEL_COLORS[name], linewidth=1.8, label=name)
    except Exception as e:
        print(f"  {name}: CF modulus failed — {e}")

u_crit = 1.0 / np.sqrt(T)
ax.axvline(x=u_crit, color=COLOR_PALETTE['gray'], linestyle='--', linewidth=1.4,
           label=f'u = 1/√T  ({u_crit:.2f})')
ax.set_xlabel('u', fontsize=12)
ax.set_ylabel('|φ(u)|', fontsize=12)
ax.set_title('Figure 3 — Characteristic Function Modulus Decay by Model', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## Section 4 — Implied Volatility Smile Comparison

The **implied volatility smile** is the market's summary of how option prices deviate from
Black-Scholes across strikes. Different models produce qualitatively distinct smiles:

- **BSM**: flat smile by construction — a single σ prices all strikes
- **Heston**: smile with term-structure mean-reversion; skew driven by correlation ρ
- **VG / CGMY / NIG**: pure-jump Lévy processes producing pronounced wings and asymmetric skew
- **MJD**: jump-driven smile; pronounced left skew from negative mean jump μ_J

**Skew** (the slope ∂σ_IV/∂K) is the primary P&L risk for option sellers — negative skew means OTM
puts are expensive relative to BSM. **Wings** (excess implied vol in the far OTM region) signal fat
tails / high kurtosis in the risk-neutral distribution.

Smiles are computed via Carr-Madan FFT pricing followed by numerical implied-volatility inversion
using a bounded root-finder (Brent's method) against the BSM closed form.

In [ ]:
strikes = np.linspace(70, 135, 60)
smile_data = {}
for name, model in models.items():
    try:
        vols = model.implied_vol_surface(strikes, method='carr_madan')
        smile_data[name] = vols
    except Exception as e:
        print(f"  {name}: smile computation failed — {e}")
        smile_data[name] = np.full(len(strikes), np.nan)

print(f"\n{'Model':<12}", end='')
for K in [80, 90, 100, 110, 120]:
    print(f"  K={K:>3}", end='')
print()
print("-" * 55)
for name, vols in smile_data.items():
    print(f"  {name:<10}", end='')
    for K in [80, 90, 100, 110, 120]:
        idx = np.argmin(np.abs(strikes - K))
        v = vols[idx]
        print(f"  {v*100:.2f}%", end='')
    print()

In [ ]:
# --- Figure 4: Main smile overlay ---
fig, ax = plt.subplots(figsize=(14, 7))

for name, vols in smile_data.items():
    mask = ~np.isnan(vols)
    ax.plot(strikes[mask], vols[mask] * 100, color=MODEL_COLORS[name],
            linewidth=2.0, label=name)

ax.axvline(x=100, color=COLOR_PALETTE['gold'], linestyle='--', linewidth=1.6,
           label='ATM (K=100)')
ax.axhline(y=20,  color=COLOR_PALETTE['gray'], linestyle='--', linewidth=1.2,
           label='ATM Vol = 20%')

ax.set_xlabel('Strike K', fontsize=12)
ax.set_ylabel('Implied Volatility (%)', fontsize=12)
ax.set_title('Figure 4 — Implied Volatility Smile by Model (T = 1.0 year)', fontsize=14)
ax.legend(fontsize=10, ncol=2)
ax.grid(True)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0f}%'))

plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 5: Smile decomposition 2x2 ---
fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)

# Panel [0,0]: Left wing K=70-100
ax00 = fig.add_subplot(gs[0, 0])
left_mask = strikes <= 100
for name, vols in smile_data.items():
    m = left_mask & ~np.isnan(vols)
    ax00.plot(strikes[m], vols[m] * 100, color=MODEL_COLORS[name], linewidth=1.8, label=name)
ax00.set_title('Left Wing (K = 70–100)', fontsize=12)
ax00.set_xlabel('Strike K')
ax00.set_ylabel('Impl. Vol (%)')
ax00.legend(fontsize=8)
ax00.grid(True)

# Panel [0,1]: Right wing K=100-135
ax01 = fig.add_subplot(gs[0, 1])
right_mask = strikes >= 100
for name, vols in smile_data.items():
    m = right_mask & ~np.isnan(vols)
    ax01.plot(strikes[m], vols[m] * 100, color=MODEL_COLORS[name], linewidth=1.8, label=name)
ax01.set_title('Right Wing (K = 100–135)', fontsize=12)
ax01.set_xlabel('Strike K')
ax01.set_ylabel('Impl. Vol (%)')
ax01.legend(fontsize=8)
ax01.grid(True)

# Panel [1,0]: Skew metric
ax10 = fig.add_subplot(gs[1, 0])
skew_metrics = {}
for name, vols in smile_data.items():
    idx90  = np.argmin(np.abs(strikes - 90))
    idx110 = np.argmin(np.abs(strikes - 110))
    v90, v110 = vols[idx90], vols[idx110]
    if not (np.isnan(v90) or np.isnan(v110) or (v90 + v110) == 0):
        skew_metrics[name] = (v90 - v110) / (v90 + v110)
    else:
        skew_metrics[name] = np.nan

s_names = list(skew_metrics.keys())
s_vals  = [skew_metrics[n] for n in s_names]
s_colors = [COLOR_PALETTE['green'] if (not np.isnan(v) and v < 0)
            else COLOR_PALETTE['red'] for v in s_vals]
ax10.bar(s_names, s_vals, color=s_colors, edgecolor='#A8BFCE', linewidth=0.8)
ax10.axhline(0, color=COLOR_PALETTE['gray'], linewidth=0.8, linestyle='--')
ax10.set_title('Skew Metric  [IV(90)−IV(110)] / [IV(90)+IV(110)]', fontsize=10)
ax10.set_ylabel('Skew Proxy')
ax10.grid(True, axis='y')

# Panel [1,1]: Kurtosis proxy
ax11 = fig.add_subplot(gs[1, 1])
kurt_metrics = {}
for name, vols in smile_data.items():
    idx80  = np.argmin(np.abs(strikes - 80))
    idx100 = np.argmin(np.abs(strikes - 100))
    idx120 = np.argmin(np.abs(strikes - 120))
    v80, v100, v120 = vols[idx80], vols[idx100], vols[idx120]
    if not (np.isnan(v80) or np.isnan(v100) or np.isnan(v120) or v100 == 0):
        kurt_metrics[name] = (v80 + v120 - 2 * v100) / v100
    else:
        kurt_metrics[name] = np.nan

k_names = list(kurt_metrics.keys())
k_vals  = [kurt_metrics[n] for n in k_names]
ax11.bar(k_names, k_vals, color=COLOR_PALETTE['teal'], edgecolor='#A8BFCE', linewidth=0.8)
ax11.axhline(0, color=COLOR_PALETTE['gray'], linewidth=0.8, linestyle='--')
ax11.set_title('Kurtosis Proxy  [IV(80)+IV(120)−2·IV(100)] / IV(100)', fontsize=10)
ax11.set_ylabel('Wing Excess Proxy')
ax11.grid(True, axis='y')

fig.suptitle('Figure 5 — Volatility Smile Decomposition: Skew and Wing Analysis',
             fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## Section 5 — Risk-Neutral Density Extraction

The **risk-neutral density (RND)** q(S_T) is the density implied by option prices under the pricing
measure Q. Two classical extraction methods are used here:

1. **Breeden-Litzenberger (1978):** The RND equals the second derivative of the call price with
   respect to strike: q(K) = e^{rT} ∂²C / ∂K²

2. **Gil-Pelaez inversion:** Given the characteristic function φ(u), the density is recovered via:
   $$f(x) = \frac{1}{\pi} \int_0^\infty \text{Re}\left[e^{-iux}\, \varphi(u)\right] du$$

The RND encodes the market's full probabilistic view of the terminal asset price. Key features to
observe across models:
- **Left skew / negative skewness**: probability mass shifted toward downside — consistent with
  equity crash risk premium
- **Fat tails (leptokurtosis)**: jump models (CGMY, NIG, MJD) exhibit heavier tails than Gaussian
- **Mode location**: typically below the forward price F due to negative skewness

In [ ]:
from fourier_martingale import risk_neutral_density

density_data = {}
for name, model in models.items():
    try:
        S_grid, dens = risk_neutral_density(model.char_func, S, r, q, T, n_points=1024)
        density_data[name] = (S_grid, dens)
    except Exception as e:
        print(f"  {name}: density extraction failed — {e}")

print(f"Densities computed for: {list(density_data.keys())}")

In [ ]:
# --- Figure 6: RN density overlay ---
fig, ax = plt.subplots(figsize=(14, 7))

for name, (S_grid, dens) in density_data.items():
    mask = (S_grid >= 60) & (S_grid <= 160)
    Sg, dg = S_grid[mask], dens[mask]
    ax.plot(Sg, dg, color=MODEL_COLORS[name], linewidth=1.8, label=name)
    if name == 'BSM':
        ax.fill_between(Sg, 0, dg, color=COLOR_PALETTE['gold'], alpha=0.08)

ax.axvline(x=F,   color=COLOR_PALETTE['gold'],  linestyle='--', linewidth=1.6,
           label=f'Forward F = {F:.2f}')
ax.axvline(x=100, color=COLOR_PALETTE['gray'],   linestyle='--', linewidth=1.2,
           label='ATM Strike K=100')

ax.set_xlabel('Terminal Asset Price S_T', fontsize=12)
ax.set_ylabel('Risk-Neutral Density q(S_T)', fontsize=12)
ax.set_title('Figure 6 — Risk-Neutral Density of Terminal Asset Price by Model', fontsize=14)
ax.legend(fontsize=10, ncol=2)
ax.set_xlim(60, 160)
ax.set_ylim(bottom=0)
ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 7: Log tail comparison ---
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 6))

for name, (S_grid, dens) in density_data.items():
    mask_l = (S_grid >= 55) & (S_grid <= 90)
    mask_r = (S_grid >= 110) & (S_grid <= 165)
    for ax, mask in [(ax_l, mask_l), (ax_r, mask_r)]:
        Sg, dg = S_grid[mask], dens[mask]
        safe = dg > 0
        if safe.any():
            ax.semilogy(Sg[safe], dg[safe], color=MODEL_COLORS[name],
                        linewidth=1.8, label=name)

ax_l.set_title('Left Tail  (S_T < 90)', fontsize=12)
ax_l.set_xlabel('S_T')
ax_l.set_ylabel('log Density')
ax_l.legend(fontsize=9)
ax_l.grid(True, which='both')

ax_r.set_title('Right Tail  (S_T > 110)', fontsize=12)
ax_r.set_xlabel('S_T')
ax_r.set_ylabel('log Density')
ax_r.legend(fontsize=9)
ax_r.grid(True, which='both')

fig.suptitle('Figure 7 — Tail Density Comparison (Log Scale)', fontsize=15)
plt.tight_layout()
plt.show()

---
## Section 6 — Carr-Madan FFT Pricing Analysis

**Carr & Madan (1999)** showed that European call prices can be expressed as a Fourier transform:

$$C(k) = \frac{e^{-\alpha k}}{\pi} \int_0^\infty e^{-iuk}\, \Psi(u)\, du$$

where k = log(K/F) is the log-moneyness and the **damped CF** is:

$$\Psi(u) = \frac{e^{-rT}\, \varphi(u - (\alpha+1)i)}{\alpha^2 + \alpha - u^2 + i(2\alpha+1)u}$$

The **dampening parameter α > 0** (typically 1.0–1.5) ensures L² integrability of the integrand.
The integral is then discretised and evaluated with the **Fast Fourier Transform** in O(N log N) time,
yielding option prices simultaneously across an entire grid of N strikes.

**Numerical parameters used:** N = 4096, Δu (eta) = 0.25, α = 1.5. The resulting strike grid has
spacing Δk = 2π/(N·Δu) ≈ 0.006, providing excellent strike resolution near the money.

In [ ]:
K_range = np.linspace(70, 130, 50)
bsm_model = models['BSM']
cf_prices  = np.array([bsm_call(S, K, r, q, T, 0.20) for K in K_range])
fft_prices = bsm_model.price_carr_madan(K_range)
abs_errors = np.abs(fft_prices - cf_prices)
rel_errors = abs_errors / np.maximum(cf_prices, 1e-6) * 100

print("Carr-Madan FFT vs Closed-Form BSM Benchmark")
print("=" * 55)
print(f"  Max absolute error:   ${abs_errors.max():.6f}")
print(f"  Mean absolute error:  ${abs_errors.mean():.6f}")
print(f"  Max relative error:   {rel_errors.max():.4f}%")
print(f"  Mean relative error:  {rel_errors.mean():.4f}%")
print(f"  Sub-penny accuracy:   {'YES' if abs_errors.max() < 0.01 else 'NO'}")

In [ ]:
# --- Figure 8: 3-panel FFT validation ---
fig = plt.figure(figsize=(18, 6))
gs  = gridspec.GridSpec(1, 3, wspace=0.35)

# Panel 0: price comparison
ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(K_range, cf_prices,  color=COLOR_PALETTE['gold'], linewidth=2.2, label='BSM Closed-Form')
ax0.plot(K_range, fft_prices, color=COLOR_PALETTE['lteal'], linewidth=1.6, linestyle='--',
         label='Carr-Madan FFT')
ax0.set_xlabel('Strike K')
ax0.set_ylabel('Option Price ($)')
ax0.set_title('Call Price Comparison', fontsize=12)
ax0.legend(fontsize=9)
ax0.grid(True)

# Panel 1: absolute errors
ax1 = fig.add_subplot(gs[0, 1])
ax1.bar(K_range, abs_errors, width=1.0, color=COLOR_PALETTE['lteal'],
        edgecolor='#A8BFCE', linewidth=0.4, label='Abs Error')
ax1.axhline(0.01, color=COLOR_PALETTE['red'], linestyle='--', linewidth=1.4,
            label='$0.01 threshold')
ax1.set_xlabel('Strike K')
ax1.set_ylabel('Absolute Error ($)')
ax1.set_title('Absolute Pricing Error', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, axis='y')

# Panel 2: relative errors
ax2 = fig.add_subplot(gs[0, 2])
ax2.bar(K_range, rel_errors, width=1.0, color=COLOR_PALETTE['gold'],
        edgecolor='#A8BFCE', linewidth=0.4, label='Rel Error (%)')
ax2.set_xlabel('Strike K')
ax2.set_ylabel('Relative Error (%)')
ax2.set_title('Relative Pricing Error', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, axis='y')

fig.suptitle('Figure 8 — Carr-Madan FFT Validation Against Black-Scholes Closed Form',
             fontsize=15)
fig.text(0.5, -0.03,
         'N=4096, alpha=1.5, eta=0.25 | BSM: S=100, r=5%, q=2%, T=1y, sigma=20%',
         ha='center', fontsize=9, color=COLOR_PALETTE['gray'])

plt.tight_layout()
plt.show()

---
## Section 7 — COS Method Convergence Study

**Fang & Oosterlee (2008)** developed the COS method, which approximates the option payoff density
on a truncated interval [a, b] using a cosine series expansion:

$$f(x) \approx \sum_{k=0}^{N-1}{}' A_k \cos\left(k\pi \frac{x-a}{b-a}\right), \quad
A_k = \frac{2}{b-a}\, \text{Re}\left[\varphi\left(\frac{k\pi}{b-a}\right) e^{-ika\pi/(b-a)}\right]$$

The key advantage over Carr-Madan FFT is **exponential convergence** in N — doubling the number of
terms roughly squares the accuracy — which allows sub-basis-point pricing accuracy with N ≈ 128–256
for smooth densities.

The truncation interval [a, b] is chosen based on the cumulants of the log-return distribution,
typically as [c₁ − L√(c₂ + √c₄), c₁ + L√(c₂ + √c₄)] with L ≈ 10–12 for high accuracy.

**Lévy models** with infinite-activity jumps (CGMY, NIG) converge slightly slower than diffusion
models because their densities are less smooth at the origin.

In [ ]:
K_atm = 100.0
N_values = [8, 16, 32, 64, 128, 256, 512]
reference_price = bsm_call(S, K_atm, r, q, T, 0.20)
cos_errors = {}

for name, model in models.items():
    errors = []
    for N in N_values:
        try:
            price = model.price_cos(np.array([K_atm]), N=N)[0]
            errors.append(abs(price - reference_price) / reference_price * 100)
        except Exception:
            errors.append(np.nan)
    cos_errors[name] = errors

print(f"{'N':>6} | ", end='')
for name in models:
    print(f"{name:>10}", end=' | ')
print()
print("-" * 80)
for i, N in enumerate(N_values):
    print(f"{N:>6} | ", end='')
    for name in models:
        v = cos_errors[name][i]
        print(f"{v:>9.4f}%" if not np.isnan(v) else f"{'N/A':>10}", end=' | ')
    print()

In [ ]:
# --- Figure 9: COS convergence ---
fig, ax = plt.subplots(figsize=(12, 6))

for name in models:
    errs = cos_errors[name]
    valid = [(N, e) for N, e in zip(N_values, errs)
             if not np.isnan(e) and e > 0]
    if valid:
        Ns, es = zip(*valid)
        ax.plot(Ns, es, color=MODEL_COLORS[name], linewidth=1.8,
                marker='o', markersize=5, label=name)

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.axhline(1.00, color=COLOR_PALETTE['gray'], linestyle='--', linewidth=1.0,
           alpha=0.8, label='1.00% threshold')
ax.axhline(0.01, color=COLOR_PALETTE['gray'], linestyle='--', linewidth=1.0,
           alpha=0.5, label='0.01% threshold')

ax.set_xlabel('Number of COS Terms N', fontsize=12)
ax.set_ylabel('Relative Pricing Error (%)', fontsize=12)
ax.set_title('Figure 9 — COS Method Convergence: Relative Pricing Error vs Number of Terms',
             fontsize=14)
ax.set_xticks(N_values)
ax.set_xticklabels([str(n) for n in N_values])
ax.legend(fontsize=10, ncol=2)
ax.grid(True, which='both', alpha=0.5)

plt.tight_layout()
plt.show()

---
## Section 8 — Volatility Surface Calibration

Calibration fits model parameters to observed market implied volatilities by minimising a
**vega-weighted RMSE** objective:

$$\mathcal{L}(\theta) = \sqrt{\frac{1}{n} \sum_{i=1}^n w_i \left(\hat{\sigma}_i(\theta) - \sigma_i^{mkt}\right)^2}$$

where weights w_i ∝ vega_i normalise across the smile. The optimisation is performed using
**Differential Evolution** (Storn & Price 1997), a global optimiser that avoids local minima
common with gradient-based methods in high-dimensional parameter spaces.

**Synthetic market data** is generated from the true Heston model with added Gaussian noise
(σ_noise = 0.1%) to simulate realistic market microstructure noise. The calibration must then
recover parameters close to the true values.

RMSE is reported in **basis points (bps)** of implied volatility, where 1 bps = 0.01%.
Market-quality fits typically achieve RMSE < 10 bps.

In [ ]:
import time

true_heston = HestonModel(S, r, q, T, v0=0.04, kappa=1.5, theta=0.05, xi=0.40, rho=-0.65)
cal_strikes = np.array([80., 85., 90., 95., 100., 105., 110., 115., 120.])
synthetic_vols = true_heston.implied_vol_surface(cal_strikes)
np.random.seed(42)
noisy_vols = synthetic_vols + np.random.normal(0, 0.001, len(cal_strikes))
noisy_vols = np.clip(noisy_vols, 0.01, 2.0)

market = MarketData(S, r, q, T, cal_strikes, noisy_vols)
engine = CalibrationEngine(market, method='differential_evolution')

print("Running Heston calibration (this may take 30-90 seconds)...")
t0 = time.time()
heston_result = engine.calibrate_heston()
elapsed = time.time() - t0
print(heston_result.summary())
print(f"\n  Calibration runtime: {elapsed:.1f}s")

In [ ]:
print("Running Variance Gamma calibration...")
t0 = time.time()
vg_result = engine.calibrate_vg()
elapsed = time.time() - t0
print(vg_result.summary())
print(f"\n  Calibration runtime: {elapsed:.1f}s")

In [ ]:
# --- Figure 10: Calibration fit quality (1x2) ---
true_vols = synthetic_vols  # from true_heston

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

results_map = {
    'Heston': (heston_result, COLOR_PALETTE['teal']),
    'VG':     (vg_result,     COLOR_PALETTE['gold']),
}

for ax, (model_label, (result, color)) in zip(axes, results_map.items()):
    ax.scatter(cal_strikes, noisy_vols * 100,
               color=COLOR_PALETTE['gold'], s=80, zorder=5,
               label='Market vols', marker='o')
    try:
        fitted = result.fitted_vols
        ax.plot(cal_strikes, np.array(fitted) * 100, color=color,
                linewidth=2.2, label=f'{model_label} fitted', zorder=4)
    except Exception as e:
        print(f"  {model_label} fitted_vols unavailable: {e}")
    ax.plot(cal_strikes, true_vols * 100, color=COLOR_PALETTE['gray'],
            linewidth=1.4, linestyle='--', label='True surface', zorder=3)
    ax.set_xlabel('Strike K')
    ax.set_ylabel('Implied Volatility (%)')
    ax.set_title(f'{model_label} Calibration Fit', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True)
    try:
        rmse_bps = result.rmse * 10000
        ax.text(0.97, 0.05, f'RMSE: {rmse_bps:.1f} bps',
                transform=ax.transAxes, ha='right', va='bottom',
                color=COLOR_PALETTE['white'], fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor=COLOR_PALETTE['navy'],
                          edgecolor=color, alpha=0.9))
    except Exception:
        pass

fig.suptitle('Figure 10 — Volatility Surface Calibration: Market Fit Quality',
             fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 11: Calibration residuals (1x2) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax_res, ax_rmse = axes

# Panel 0: residuals bar chart in bps
x = np.arange(len(cal_strikes))
width = 0.35

try:
    heston_resid = (np.array(heston_result.fitted_vols) - noisy_vols) * 10000
    ax_res.bar(x - width/2, heston_resid, width=width, color=COLOR_PALETTE['teal'],
               edgecolor='#A8BFCE', linewidth=0.6, label='Heston')
except Exception:
    heston_resid = np.zeros(len(cal_strikes))

try:
    vg_resid = (np.array(vg_result.fitted_vols) - noisy_vols) * 10000
    ax_res.bar(x + width/2, vg_resid, width=width, color=COLOR_PALETTE['gold'],
               edgecolor='#A8BFCE', linewidth=0.6, label='VG')
except Exception:
    vg_resid = np.zeros(len(cal_strikes))

ax_res.axhline( 5, color=COLOR_PALETTE['red'], linestyle='--', linewidth=1.2, alpha=0.7)
ax_res.axhline(-5, color=COLOR_PALETTE['red'], linestyle='--', linewidth=1.2, alpha=0.7,
               label='±5 bps band')
ax_res.axhline(0,  color=COLOR_PALETTE['gray'], linewidth=0.8)
ax_res.set_xticks(x)
ax_res.set_xticklabels([f'{int(k)}' for k in cal_strikes], rotation=45)
ax_res.set_xlabel('Strike K')
ax_res.set_ylabel('Residual (bps)')
ax_res.set_title('Calibration Residuals by Strike', fontsize=12)
ax_res.legend(fontsize=9)
ax_res.grid(True, axis='y')

# Panel 1: RMSE horizontal bar comparison
rmse_vals = {}
for lbl, res in [('Heston', heston_result), ('VG', vg_result)]:
    try:
        rmse_vals[lbl] = res.rmse * 10000
    except Exception:
        rmse_vals[lbl] = np.nan

rmse_names  = list(rmse_vals.keys())
rmse_bps    = [rmse_vals[n] for n in rmse_names]
rmse_colors = []
for v in rmse_bps:
    if np.isnan(v):
        rmse_colors.append(COLOR_PALETTE['gray'])
    elif v < 10:
        rmse_colors.append(COLOR_PALETTE['green'])
    elif v < 25:
        rmse_colors.append(COLOR_PALETTE['gold'])
    else:
        rmse_colors.append(COLOR_PALETTE['red'])

ax_rmse.barh(rmse_names, rmse_bps, color=rmse_colors,
             edgecolor='#A8BFCE', linewidth=0.8, height=0.5)
ax_rmse.axvline(25, color=COLOR_PALETTE['red'], linestyle='--',
                linewidth=1.4, label='25 bps threshold')
ax_rmse.set_xlabel('RMSE (bps)')
ax_rmse.set_title('RMSE Comparison by Model', fontsize=12)
ax_rmse.legend(fontsize=9)
ax_rmse.grid(True, axis='x')

for bar, val in zip(ax_rmse.patches, rmse_bps):
    if not np.isnan(val):
        ax_rmse.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                     f'{val:.2f} bps', va='center', fontsize=10,
                     color=COLOR_PALETTE['white'])

green_p = mpatches.Patch(color=COLOR_PALETTE['green'], label='< 10 bps (Excellent)')
gold_p  = mpatches.Patch(color=COLOR_PALETTE['gold'],  label='10-25 bps (Good)')
red_p   = mpatches.Patch(color=COLOR_PALETTE['red'],   label='> 25 bps (Poor)')
ax_rmse.legend(handles=[green_p, gold_p, red_p], fontsize=8, loc='lower right')

fig.suptitle('Figure 11 — Calibration Residual Analysis', fontsize=15)
plt.tight_layout()
plt.show()

---
## Section 9 — Fourier Risk Analytics

The characteristic function enables direct computation of **risk measures** and **option Greeks**
without Monte Carlo simulation:

### Value-at-Risk via Gil-Pelaez Inversion
The α-VaR (log-return) satisfies F(VaR_α) = α, where F is the CDF recovered from φ:
$$F(x) = \frac{1}{2} - \frac{1}{\pi}\int_0^\infty \frac{\text{Re}[e^{-iux}\varphi(u)]}{u}\, du$$
Numerical inversion is performed via bisection.

### Greeks via Fourier Finite Differences
- **Delta** = ∂C/∂S ≈ [C(S+ε) − C(S−ε)] / (2ε)
- **Gamma** = ∂²C/∂S² ≈ [C(S+ε) − 2C(S) + C(S−ε)] / ε²
- **Theta** = ∂C/∂T ≈ [C(T+ε) − C(T−ε)] / (2ε)

Each finite-difference evaluation calls the FFT pricer once, making the total computational cost
O(N log N) per Greek — the same order as a single vanilla price.

In [ ]:
heston_model = HestonModel(S, r, q, T, v0=0.04, kappa=2.0, theta=0.04, xi=0.3, rho=-0.70)
analytics    = FourierRiskAnalytics(heston_model)
alphas = [0.01, 0.025, 0.05, 0.10]

print("Value-at-Risk Analysis (Heston Model)")
print("=" * 55)
print(f"  {'Confidence':>12}  {'alpha':>6}  {'Log VaR':>10}  {'Gross Return':>14}")
print("-" * 55)
for alpha in alphas:
    var_val = analytics.var(alpha=alpha)
    gross   = (np.exp(var_val) - 1) * 100
    print(f"  {(1-alpha)*100:>11.1f}%  {alpha:>6.3f}  {var_val:>10.4f}  {gross:>13.2f}%")

In [ ]:
greek_strikes = np.array([85., 90., 95., 100., 105., 110., 115.])
greeks_list = []
for K in greek_strikes:
    try:
        g = analytics.compute_greeks_fourier(K)
        greeks_list.append({'K': K, **g})
    except Exception as e:
        print(f"  K={K}: {e}")
        greeks_list.append({'K': K, 'delta': np.nan, 'gamma': np.nan, 'theta': np.nan})

greeks_df = pd.DataFrame(greeks_list).set_index('K')
print("\nHeston Model Greeks (Fourier-based)")
print(greeks_df.round(4).to_string())

In [ ]:
# --- Figure 12: Risk analytics dashboard (2x2) ---
bsm_analytics = FourierRiskAnalytics(models['BSM'])

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

# Panel [0,0]: VaR bar chart BSM vs Heston
ax00 = fig.add_subplot(gs[0, 0])
x_alpha = np.arange(len(alphas))
width   = 0.35
bsm_vars    = []
heston_vars = []
for a in alphas:
    try:
        bsm_vars.append((np.exp(bsm_analytics.var(alpha=a)) - 1) * 100)
    except Exception:
        bsm_vars.append(np.nan)
    try:
        heston_vars.append((np.exp(analytics.var(alpha=a)) - 1) * 100)
    except Exception:
        heston_vars.append(np.nan)

ax00.bar(x_alpha - width/2, bsm_vars,    width=width, color=MODEL_COLORS['BSM'],
         edgecolor='#A8BFCE', linewidth=0.6, label='BSM')
ax00.bar(x_alpha + width/2, heston_vars, width=width, color=MODEL_COLORS['Heston'],
         edgecolor='#A8BFCE', linewidth=0.6, label='Heston')
ax00.set_xticks(x_alpha)
ax00.set_xticklabels([f'{int((1-a)*100)}%' for a in alphas])
ax00.set_xlabel('Confidence Level')
ax00.set_ylabel('Gross Return VaR (%)')
ax00.set_title('VaR Comparison: BSM vs Heston', fontsize=12)
ax00.legend(fontsize=10)
ax00.grid(True, axis='y')

# Panel [0,1]: Heston RN density with VaR regions
ax01 = fig.add_subplot(gs[0, 1])
if 'Heston' in density_data:
    Sg_h, dg_h = density_data['Heston']
    mask_d = (Sg_h >= 55) & (Sg_h <= 165)
    ax01.plot(Sg_h[mask_d], dg_h[mask_d], color=MODEL_COLORS['Heston'],
              linewidth=2.0, label='Heston RND')
    alpha_fill = 0.05
    try:
        var_05 = analytics.var(alpha=alpha_fill)
        var_price = S * np.exp(var_05)
        tail_mask = mask_d & (Sg_h <= var_price)
        ax01.fill_between(Sg_h[tail_mask], 0, dg_h[tail_mask],
                          color=COLOR_PALETTE['red'], alpha=0.4,
                          label=f'VaR {int((1-alpha_fill)*100)}% tail')
        ax01.axvline(x=var_price, color=COLOR_PALETTE['red'],
                     linestyle='--', linewidth=1.4)
    except Exception:
        pass
ax01.axvline(x=F, color=COLOR_PALETTE['gold'], linestyle='--',
             linewidth=1.4, label=f'Forward F={F:.1f}')
ax01.set_xlabel('S_T')
ax01.set_ylabel('Density')
ax01.set_title('Heston RND with VaR Tail Region', fontsize=12)
ax01.legend(fontsize=9)
ax01.grid(True)
ax01.set_xlim(55, 165)
ax01.set_ylim(bottom=0)

# Panel [1,0]: Delta across strikes
ax10 = fig.add_subplot(gs[1, 0])
deltas = greeks_df['delta'].values
ax10.plot(greek_strikes, deltas, color=MODEL_COLORS['Heston'],
          linewidth=2.2, marker='o', markersize=6, label='Delta')
ax10.axhline(0.5, color=COLOR_PALETTE['gray'], linestyle='--',
             linewidth=1.0, alpha=0.7, label='Delta=0.5 (ATM)')
ax10.axvline(100, color=COLOR_PALETTE['gold'], linestyle='--',
             linewidth=1.0, alpha=0.7)
ax10.set_xlabel('Strike K')
ax10.set_ylabel('Delta')
ax10.set_title('Delta vs Strike (Heston)', fontsize=12)
ax10.legend(fontsize=9)
ax10.grid(True)

# Panel [1,1]: Gamma across strikes
ax11 = fig.add_subplot(gs[1, 1])
gammas = greeks_df['gamma'].values
ax11.plot(greek_strikes, gammas, color=COLOR_PALETTE['gold'],
          linewidth=2.2, marker='s', markersize=6, label='Gamma')
ax11.fill_between(greek_strikes, 0, gammas,
                  color=COLOR_PALETTE['gold'], alpha=0.15)
# Annotate max gamma
valid_gamma = ~np.isnan(gammas)
if valid_gamma.any():
    max_idx = np.nanargmax(gammas)
    ax11.annotate(f'Max Γ={gammas[max_idx]:.4f}',
                  xy=(greek_strikes[max_idx], gammas[max_idx]),
                  xytext=(greek_strikes[max_idx] + 3, gammas[max_idx] * 0.9),
                  arrowprops=dict(arrowstyle='->', color=COLOR_PALETTE['white']),
                  color=COLOR_PALETTE['white'], fontsize=9)
ax11.set_xlabel('Strike K')
ax11.set_ylabel('Gamma')
ax11.set_title('Gamma vs Strike (Heston)', fontsize=12)
ax11.legend(fontsize=9)
ax11.grid(True)

fig.suptitle('Figure 12 — Fourier Risk Analytics Dashboard (Heston Model, T=1.0 year)',
             fontsize=15)
plt.tight_layout()
plt.show()

---
## Section 10 — Cumulant Analysis

**Cumulants** of the log-return distribution encode its distributional shape in a compact form.
For a random variable X with log-characteristic function Ψ(u) = log φ(u), the cumulants are:

$$\kappa_n = \frac{1}{i^n} \left.\frac{d^n \Psi(u)}{du^n}\right|_{u=0}$$

| Cumulant | Symbol | Interpretation |
|----------|--------|----------------|
| 1st | κ₁ = μ | Mean log-return (drift) |
| 2nd | κ₂ = σ² | Variance (annualised vol² = κ₂/T) |
| 3rd | κ₃ | Raw skewness; negative → left tail |
| 4th | κ₄ | Excess kurtosis proxy; positive → fat tails |

**Model-consistent extraction:** Rather than simulating, cumulants are extracted analytically via
numerical differentiation of log φ(u) at u = 0. This guarantees consistency with the pricing CF.

A key theoretical result: for Lévy processes, cumulants scale **linearly with T**, so annualised
skewness scales as T^{−1/2} and excess kurtosis scales as T^{−1}.

In [ ]:
cumulants = {}
for name, model in models.items():
    try:
        c = MartingaleDiagnostics.cumulant_analysis(model)
        cumulants[name] = c
    except Exception as e:
        print(f"  {name}: cumulant extraction failed — {e}")

cum_df = pd.DataFrame(cumulants).T
cum_df.index.name = 'Model'
print("\nCumulant Analysis — Log-Return Distribution (T=1 year)")
print("=" * 70)
print(cum_df.round(4).to_string())

In [ ]:
# --- Figure 13: Cumulant comparison (2x2) ---
cum_names = list(cumulants.keys())
bar_colors_models = [MODEL_COLORS[n] for n in cum_names]

def safe_get(d, key, default=np.nan):
    try:
        return float(d.get(key, default))
    except Exception:
        return default

means     = [safe_get(cumulants[n], 'mean')     for n in cum_names]
variances = [safe_get(cumulants[n], 'variance') for n in cum_names]
skews     = [safe_get(cumulants[n], 'skewness') for n in cum_names]
kurts     = [safe_get(cumulants[n], 'kurtosis') for n in cum_names]

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)

# Panel [0,0]: Mean log-return
ax00 = fig.add_subplot(gs[0, 0])
ax00.barh(cum_names, means, color=bar_colors_models,
          edgecolor='#A8BFCE', linewidth=0.8)
ax00.axvline(0, color=COLOR_PALETTE['gray'], linewidth=0.8, linestyle='--')
ax00.set_xlabel('Mean Log-Return')
ax00.set_title('κ₁ — Mean Log-Return', fontsize=12)
ax00.grid(True, axis='x')

# Panel [0,1]: Variance with vol annotation
ax01 = fig.add_subplot(gs[0, 1])
ax01.barh(cum_names, variances, color=bar_colors_models,
          edgecolor='#A8BFCE', linewidth=0.8)
ax01.set_xlabel('Variance κ₂')
ax01.set_title('κ₂ — Variance (annualised σ = √κ₂)', fontsize=12)
for i, (name, v) in enumerate(zip(cum_names, variances)):
    if not np.isnan(v) and v > 0:
        vol = np.sqrt(v) * 100
        ax01.text(v + 1e-4, i, f'σ={vol:.1f}%',
                  va='center', ha='left', fontsize=8,
                  color=COLOR_PALETTE['white'])
ax01.grid(True, axis='x')

# Panel [1,0]: Skewness
ax10 = fig.add_subplot(gs[1, 0])
sk_colors = [COLOR_PALETTE['green'] if (not np.isnan(v) and v < 0)
             else COLOR_PALETTE['red'] for v in skews]
ax10.barh(cum_names, skews, color=sk_colors, edgecolor='#A8BFCE', linewidth=0.8)
ax10.axvline(0, color=COLOR_PALETTE['gray'], linewidth=1.0, linestyle='--')
ax10.set_xlabel('Skewness κ₃')
ax10.set_title('κ₃ — Skewness (−ve = left tail)', fontsize=12)
ax10.grid(True, axis='x')
green_p = mpatches.Patch(color=COLOR_PALETTE['green'], label='Negative (left skew)')
red_p   = mpatches.Patch(color=COLOR_PALETTE['red'],   label='Positive (right skew)')
ax10.legend(handles=[green_p, red_p], fontsize=8, loc='lower right')

# Panel [1,1]: Excess kurtosis (truncate at 50)
ax11 = fig.add_subplot(gs[1, 1])
kurt_display = [min(v, 50) if not np.isnan(v) else np.nan for v in kurts]
ax11.barh(cum_names, kurt_display, color=COLOR_PALETTE['gold'],
          edgecolor='#A8BFCE', linewidth=0.8)
ax11.axvline(0, color=COLOR_PALETTE['gray'], linewidth=1.0, linestyle='--')
ax11.set_xlabel('Excess Kurtosis κ₄  (capped at 50)')
ax11.set_title('κ₄ — Excess Kurtosis (fat tails)', fontsize=12)
for i, (v_disp, v_real) in enumerate(zip(kurt_display, kurts)):
    if not np.isnan(v_real) and v_real > 50:
        ax11.text(50.5, i, f'{v_real:.1f}→',
                  va='center', ha='left', fontsize=8,
                  color=COLOR_PALETTE['orange'])
ax11.grid(True, axis='x')

fig.suptitle('Figure 13 — Cumulant Analysis: Distributional Properties by Model',
             fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 14: Summary heatmap (6 models x 6 metrics) ---
model_list = list(models.keys())

# Build metrics matrix
metric_labels = [
    'ATM Vol (%)',
    'Smile Skew (%)',
    'Wing Excess (%)',
    'Skewness',
    'Excess Kurtosis',
    'Martingale Err (log10)',
]

raw_matrix = np.full((len(model_list), len(metric_labels)), np.nan)

for i, name in enumerate(model_list):
    vols = smile_data.get(name, np.full(len(strikes), np.nan))

    # ATM Vol
    idx_atm = np.argmin(np.abs(strikes - 100))
    raw_matrix[i, 0] = vols[idx_atm] * 100 if not np.isnan(vols[idx_atm]) else np.nan

    # Smile skew proxy
    idx90  = np.argmin(np.abs(strikes - 90))
    idx110 = np.argmin(np.abs(strikes - 110))
    v90, v110 = vols[idx90], vols[idx110]
    if not (np.isnan(v90) or np.isnan(v110) or (v90 + v110) == 0):
        raw_matrix[i, 1] = (v90 - v110) / (v90 + v110) * 100

    # Wing excess proxy
    idx80  = np.argmin(np.abs(strikes - 80))
    idx120 = np.argmin(np.abs(strikes - 120))
    v80, v100, v120 = vols[idx80], vols[idx_atm], vols[idx120]
    if not (np.isnan(v80) or np.isnan(v100) or np.isnan(v120) or v100 == 0):
        raw_matrix[i, 2] = (v80 + v120 - 2 * v100) / v100 * 100

    # Skewness, kurtosis
    if name in cumulants:
        raw_matrix[i, 3] = safe_get(cumulants[name], 'skewness')
        raw_matrix[i, 4] = safe_get(cumulants[name], 'kurtosis')

    # Martingale error (log10)
    if name in mart_errors and not np.isnan(mart_errors[name]) and mart_errors[name] > 0:
        raw_matrix[i, 5] = np.log10(mart_errors[name])
    else:
        raw_matrix[i, 5] = -15.0  # effectively machine eps

# Normalise each column to [0, 1]
norm_matrix = np.zeros_like(raw_matrix)
for j in range(raw_matrix.shape[1]):
    col = raw_matrix[:, j]
    valid = col[~np.isnan(col)]
    if len(valid) > 0 and valid.max() != valid.min():
        norm_matrix[:, j] = (col - valid.min()) / (valid.max() - valid.min())
    else:
        norm_matrix[:, j] = 0.5

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(norm_matrix, cmap='Blues', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(np.arange(len(metric_labels)))
ax.set_yticks(np.arange(len(model_list)))
ax.set_xticklabels(metric_labels, rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(model_list, fontsize=11)

# Annotate each cell with raw value
for i in range(len(model_list)):
    for j in range(len(metric_labels)):
        val = raw_matrix[i, j]
        if np.isnan(val):
            txt = 'N/A'
        elif j == 5:  # log10 scale
            txt = f'{val:.1f}'
        else:
            txt = f'{val:.2f}'
        cell_bright = norm_matrix[i, j]
        txt_color   = '#0D1B35' if cell_bright > 0.55 else '#FFFFFF'
        ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                color=txt_color, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Normalised Value [0–1]', fontsize=10)
cbar.ax.yaxis.set_tick_params(color='#A8BFCE')

ax.set_title('Figure 14 — Model Comparison Heatmap: Normalised Metrics Summary',
             fontsize=14, pad=15)

plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "=" * 70)
print("  NOTEBOOK EXECUTION COMPLETE")
print("=" * 70)
print(f"\n  Figures generated: 14")
print(f"  Models validated:  {sum(m.verify_martingale_condition() for m in models.values())} / {len(models)}")
print(f"  Calibration RMSE (Heston): {heston_result.rmse*10000:.2f} bps")
print(f"  Calibration RMSE (VG):     {vg_result.rmse*10000:.2f} bps")
print(f"  Heston ATM Greeks:  Delta={greeks_df.loc[100, 'delta']:.4f}, Gamma={greeks_df.loc[100, 'gamma']:.4f}")
print("\n  All martingale conditions satisfied: "
      + ("YES" if all(m.verify_martingale_condition() for m in models.values()) else "NO"))
print("=" * 70)